# FINAL — Generate Dataset dan Grid Search CNN Only

Notebook ini disusun untuk skenario **Invariant Noise-Statistics and Fading**.

Spesifikasi skenario:

1. Model yang dilatih hanya **CNN**.
2. Noise power tetap: **80 mW**.
3. Fading per window bersifat **konstan / invariant**.
4. Variasi kanal hanya berasal dari path loss: **-2, -4, -6, dan -8 dB**.
5. Untuk setiap kombinasi `PU × path loss`, dibuat **2000 data**, terdiri atas:
   - 1000 data PU + noise
   - 1000 data noise-only
6. Untuk PU 1, PU 2, dan PU 3, total dataset adalah:

```text
3 PU × 4 path loss × 2000 data = 24.000 data
```

Notebook ini sudah dibuat agar dapat dijalankan **sekali dari atas ke bawah** di Google Colab atau Jupyter.


## 1. Import dan Random Seed


In [1]:
import os
import json
import math
import random
import gc
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.linalg import convolution_matrix

from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as err:
    print("Determinism tidak dapat diaktifkan penuh:", err)


## 2. Konfigurasi Umum

Ubah `BASE_DIR` sesuai lokasi penyimpanan notebook dan file `fading-10.csv` sampai `fading-15.csv`.

Contoh jika memakai Google Drive di Colab:

```python
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/nama_folder_dataset')
```

Jika file CSV fading diunggah langsung ke Colab, `BASE_DIR = Path('.')` sudah cukup.


In [2]:
BASE_DIR = Path("dataset")
OUTPUT_DIR = BASE_DIR / "generated_datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proteksi file output.
# False: jika file .npz sudah ada, skenario tersebut akan dilewati agar tidak tertimpa.
# True : file lama akan ditimpa.
OVERWRITE_DATASETS = False

COMMON_CONFIG = {
    "num_samples": 1000,
    "N1": 1000,
    "N2": 1000,
    "unfiltered_signal_length": 10000,
    "mean": 0,
    "unfiltered_signal_var": 80,
    "total_antennas": 6,
    "window_length": 1000,
    "hop": 1000,
    "divide_matrix": True,
    "corrcoef": False,
    "dtype": "float32",
}

# Tiga PU yang dibangkitkan pada sinyal sumber.
theta_ranges_1 = [
    (np.pi * 0.1, np.pi * 0.25),  # PU 1
    (np.pi * 0.4, np.pi * 0.55),  # PU 2
    (np.pi * 0.7, np.pi * 0.85),  # PU 3
]

# Tiga band penerima/detektor. Dataset akan dibuat untuk semuanya.
RECEIVER_BANDS = {
    "pu1_low":  [(np.pi * 0.08,  np.pi * 0.252)],
    "pu2_mid":  [(np.pi * 0.398, np.pi * 0.552)],
    "pu3_high": [(np.pi * 0.698, np.pi * 0.852)],
}

# Jika hanya ingin membuat sebagian PU, isi daftar ini, misalnya ["pu3_high"].
# Default None berarti seluruh PU 1, PU 2, dan PU 3 diproses.
ACTIVE_RECEIVER_BANDS = None


## 3. Daftar Skenario Dataset

Bagian ini mendefinisikan skenario **Invariant Noise-Statistics and Fading**.

Pada skenario ini:

- `noise_var` bernilai tetap 80.
- `fading` bernilai tetap dalam satu skenario.
- Path loss yang diuji adalah -2, -4, -6, dan -8 dB.
- Dataset yang dibuat hanya dataset CNN.


In [3]:
# ============================================================
# SKENARIO: Invariant Noise-Statistics and Fading
# ============================================================

SCENARIO_TYPE_INVARIANT = "invariant_noise_statistics_and_fading"

# Noise power tetap pada seluruh window.
FIXED_NOISE_VAR_INVARIANT = 80

# Path loss yang diuji.
PATH_LOSS_DB_LIST = [-2, -4, -6, -8]


def path_loss_db_to_power_gain(path_loss_db):
    """
    Mengubah path loss dalam dB menjadi gain daya linear.

    Konversi daya menggunakan 10^(dB/10). Nilai ini kemudian digunakan
    sebagai variansi/skala daya untuk membangkitkan koefisien fading.
    """
    return 10 ** (float(path_loss_db) / 10)


def generate_fading(total_antennas, path_loss):
    """
    Generate koefisien fading berdasarkan path loss.

    Alur:
    1. path_loss dB dikonversi menjadi path_loss_linear = 10^(path_loss/10)
    2. standar deviasi fading = sqrt(path_loss_linear)
    3. setiap antena/SU memperoleh koefisien fading dari N(0, sqrt(path_loss_linear))

    Fungsi ini menghasilkan satu vektor fading untuk satu skenario. Untuk
    skenario invariant, fungsi ini dipanggil sekali per skenario path loss,
    lalu vektor hasilnya dipakai tetap pada seluruh window/sample dataset.
    """
    path_loss_linear = np.tile([path_loss_db_to_power_gain(path_loss)], total_antennas)
    fading = []

    for i in range(total_antennas):
        fading.append(np.random.normal(0, math.sqrt(path_loss_linear[i]), 1)[0])

    return np.asarray(fading, dtype=np.float64)


def make_invariant_fading_from_path_loss(path_loss_db, total_antennas):
    """
    Membuat satu vektor fading acak berbasis path loss untuk satu skenario.

    Karena skenario ini invariant, vektor fading yang sudah dibangkitkan
    tidak dibangkitkan ulang di setiap window.
    """
    return generate_fading(total_antennas=total_antennas, path_loss=path_loss_db)


## 4. Fungsi Sinyal, Noise, Fading, dan Matriks Kovarian


In [4]:
def create_bpf(theta_1, theta_2, N):
    bpf = np.zeros(N)
    for i in range(N):
        bpf[i] = (
            (theta_2 / np.pi) * np.sinc(theta_2 * (i - 0.5 * N) / np.pi)
            - (theta_1 / np.pi) * np.sinc(theta_1 * (i - 0.5 * N) / np.pi)
        )
    return bpf


def generate_pu_signal(theta_ranges, mean, unfiltered_signal_var, unfiltered_signal_length, N, total_antennas):
    rand_sig = np.random.normal(mean, math.sqrt(unfiltered_signal_var), unfiltered_signal_length)
    pu_total = np.zeros(unfiltered_signal_length)

    for theta1, theta2 in theta_ranges:
        bpf = create_bpf(theta1, theta2, N)
        filtered = np.convolve(rand_sig, bpf, mode="same")
        pu_total += filtered

    return np.tile(pu_total, (total_antennas, 1))


def matrix_divider(signal_matrix):
    """
    Membagi matriks sinyal menjadi dua bagian kolom, lalu menumpuknya.

    - signal_odd  : kolom indeks 0, 2, 4, ...
    - signal_even : kolom indeks 1, 3, 5, ...

    Nama odd/even mengikuti struktur kode lama. Secara indeks Python,
    signal_odd mengambil indeks genap karena indexing dimulai dari 0.
    Urutan stack mengikuti eksperimen utama sebelumnya: even di atas odd.
    """
    signal_odd = signal_matrix[:, ::2]
    signal_even = signal_matrix[:, 1::2]

    min_cols = min(signal_odd.shape[1], signal_even.shape[1])
    signal_odd = signal_odd[:, :min_cols]
    signal_even = signal_even[:, :min_cols]

    return np.vstack((signal_even, signal_odd))


def generate_gaussian_noise(n, mean, noise_var, length):
    return np.random.normal(mean, math.sqrt(float(noise_var)), size=(n, length))


def load_fading_csv(csv_path):
    df = pd.read_csv(csv_path)
    fading = np.array([df[col].to_numpy() for col in df.columns], dtype=np.float64)

    # Format yang umum dari notebook lama adalah (jumlah_window, total_antennas).
    # Jika CSV terbaca sebagai (total_antennas, jumlah_window), transpose otomatis.
    if fading.shape[0] == COMMON_CONFIG["total_antennas"] and fading.shape[1] != COMMON_CONFIG["total_antennas"]:
        fading = fading.T

    return fading


def get_noise_for_window(noise_var, window_index):
    if np.isscalar(noise_var):
        return float(noise_var)
    return float(noise_var[window_index])


def get_fading_for_window(fading, window_index, total_antennas):
    fading = np.asarray(fading)

    if fading.ndim == 1:
        if fading.shape[0] != total_antennas:
            raise ValueError(f"Fading tetap harus berukuran ({total_antennas},), tetapi diperoleh {fading.shape}")
        return fading

    if fading.ndim == 2:
        if fading.shape[0] <= window_index:
            raise ValueError(f"Jumlah baris fading ({fading.shape[0]}) lebih kecil dari jumlah window yang dibutuhkan.")
        if fading.shape[1] != total_antennas:
            raise ValueError(f"Fading varying harus berukuran (jumlah_window, {total_antennas}), tetapi diperoleh {fading.shape}")
        return fading[window_index]

    raise ValueError(f"Format fading tidak dikenali: {fading.shape}")


def apply_fading(segment, fading_vector):
    # Pada notebook lama fading digunakan melalui np.convolve.
    # Jika nilai fading adalah skalar per antena, operasi tersebut ekuivalen dengan perkalian amplitudo.
    return segment * fading_vector[:, np.newaxis]


def build_receiver_filter(theta_ranges_2, N2, window_length):
    bpf_total = np.zeros(N2)
    for theta_1, theta_2 in theta_ranges_2:
        bpf_total += create_bpf(theta_1, theta_2, N2)
    return convolution_matrix(bpf_total, window_length, mode="same")


def covariance_matrix(signal_matrix, divide_matrix=True, corrcoef=False):
    if divide_matrix:
        signal_matrix = matrix_divider(signal_matrix)

    if corrcoef:
        return np.corrcoef(signal_matrix)

    centered = signal_matrix - signal_matrix.mean(axis=1, keepdims=True)
    return (centered @ centered.T) / centered.shape[1]


## 5. Fungsi Pembentukan Dataset CNN dan CNN-LSTM


In [5]:
def generate_windowed_data(signal, window_length, hop, mean, noise_var, fading):
    A, T = signal.shape
    starts = list(range(0, T - window_length + 1, hop))

    if not np.isscalar(noise_var) and len(noise_var) != len(starts):
        raise ValueError(f"noise_var harus skalar atau berisi {len(starts)} nilai, tetapi diperoleh {len(noise_var)} nilai.")

    windows_pu = []
    windows_noise = []

    for window_index, start in enumerate(starts):
        pu_segment = signal[:, start:start + window_length]

        current_noise_var = get_noise_for_window(noise_var, window_index)
        noise_full = generate_gaussian_noise(A, mean, current_noise_var, T)
        noise_segment = noise_full[:, start:start + window_length]

        fading_vector = get_fading_for_window(fading, window_index, A)
        faded_segment = apply_fading(pu_segment, fading_vector)

        windows_pu.append(faded_segment + noise_segment)
        windows_noise.append(noise_segment)

    return windows_pu, windows_noise


def filter_windows(windows, H):
    W = len(windows)
    A = windows[0].shape[0]

    columns = []
    for w in range(W):
        for a in range(A):
            columns.append(windows[w][a])

    X = np.column_stack(columns)
    Y = H @ X

    windows_filtered = []
    idx = 0
    for w in range(W):
        seg = []
        for a in range(A):
            seg.append(Y[:, idx])
            idx += 1
        windows_filtered.append(np.array(seg))

    return windows_filtered


def make_cnn_matrix(windows, H, divide_matrix=True, corrcoef=False):
    windows_filtered = filter_windows(windows, H)
    signal_filtered = np.hstack(windows_filtered)
    return covariance_matrix(signal_filtered, divide_matrix=divide_matrix, corrcoef=corrcoef)


def make_cnn_lstm_sequence(windows, H, divide_matrix=True, corrcoef=False):
    windows_filtered = filter_windows(windows, H)
    corr_seq = []
    for seg in windows_filtered:
        corr_seq.append(covariance_matrix(seg, divide_matrix=divide_matrix, corrcoef=corrcoef))
    return np.array(corr_seq)


def create_dataset_pair(
    num_samples,
    N1,
    N2,
    unfiltered_signal_length,
    mean,
    unfiltered_signal_var,
    noise_var,
    total_antennas,
    theta_ranges_1,
    theta_ranges_2,
    window_length,
    hop,
    fading,
    divide_matrix=True,
    corrcoef=False,
    dtype="float32",
):
    H = build_receiver_filter(theta_ranges_2, N2, window_length)

    X_cnn, y_cnn = [], []
    X_cnn_lstm, y_cnn_lstm = [], []

    for sample_index in range(num_samples):
        if (sample_index + 1) % 100 == 0:
            print(f"  sample {sample_index + 1}/{num_samples}")

        PU_Signal = generate_pu_signal(
            theta_ranges_1,
            mean,
            unfiltered_signal_var,
            unfiltered_signal_length,
            N1,
            total_antennas,
        )

        shared_windows_pu, shared_windows_noise = generate_windowed_data(
            PU_Signal,
            window_length,
            hop,
            mean,
            noise_var,
            fading,
        )

        corr_pu_cnn = make_cnn_matrix(shared_windows_pu, H, divide_matrix=divide_matrix, corrcoef=corrcoef)
        corr_noise_cnn = make_cnn_matrix(shared_windows_noise, H, divide_matrix=divide_matrix, corrcoef=corrcoef)

        corr_pu_cnn_lstm = make_cnn_lstm_sequence(shared_windows_pu, H, divide_matrix=divide_matrix, corrcoef=corrcoef)
        corr_noise_cnn_lstm = make_cnn_lstm_sequence(shared_windows_noise, H, divide_matrix=divide_matrix, corrcoef=corrcoef)

        X_cnn.extend([corr_pu_cnn, corr_noise_cnn])
        y_cnn.extend([1, 0])

        X_cnn_lstm.extend([corr_pu_cnn_lstm, corr_noise_cnn_lstm])
        y_cnn_lstm.extend([1, 0])

    X_cnn = np.asarray(X_cnn, dtype=dtype)[..., np.newaxis]
    y_cnn = np.asarray(y_cnn, dtype=np.int64)

    X_cnn_lstm = np.asarray(X_cnn_lstm, dtype=dtype)[..., np.newaxis]
    y_cnn_lstm = np.asarray(y_cnn_lstm, dtype=np.int64)

    return {
        "cnn": {"X": X_cnn, "y": y_cnn},
        "cnn_lstm": {"X": X_cnn_lstm, "y": y_cnn_lstm},
    }


# ============================================================
# Dataset CNN-only untuk skenario invariant
# ============================================================

def create_dataset_cnn_only(
    num_samples,
    N1,
    N2,
    unfiltered_signal_length,
    mean,
    unfiltered_signal_var,
    noise_var,
    total_antennas,
    theta_ranges_1,
    theta_ranges_2,
    window_length,
    hop,
    fading,
    divide_matrix=True,
    corrcoef=False,
    dtype="float32",
):
    H = build_receiver_filter(theta_ranges_2, N2, window_length)

    X_cnn, y_cnn = [], []

    for sample_index in range(num_samples):
        if (sample_index + 1) % 100 == 0:
            print(f"  sample {sample_index + 1}/{num_samples}")

        PU_Signal = generate_pu_signal(
            theta_ranges_1,
            mean,
            unfiltered_signal_var,
            unfiltered_signal_length,
            N1,
            total_antennas,
        )

        shared_windows_pu, shared_windows_noise = generate_windowed_data(
            PU_Signal,
            window_length,
            hop,
            mean,
            noise_var,
            fading,
        )

        corr_pu_cnn = make_cnn_matrix(
            shared_windows_pu,
            H,
            divide_matrix=divide_matrix,
            corrcoef=corrcoef,
        )

        corr_noise_cnn = make_cnn_matrix(
            shared_windows_noise,
            H,
            divide_matrix=divide_matrix,
            corrcoef=corrcoef,
        )

        X_cnn.extend([corr_pu_cnn, corr_noise_cnn])
        y_cnn.extend([1, 0])

    X_cnn = np.asarray(X_cnn, dtype=dtype)[..., np.newaxis]
    y_cnn = np.asarray(y_cnn, dtype=np.int64)

    return {
        "cnn": {
            "X": X_cnn,
            "y": y_cnn,
        }
    }


## 6. Membuat Dataset Invariant CNN-only dan Menyimpannya ke `.npz`


In [6]:
def selected_receiver_bands():
    if ACTIVE_RECEIVER_BANDS is None:
        return RECEIVER_BANDS

    missing = [band for band in ACTIVE_RECEIVER_BANDS if band not in RECEIVER_BANDS]
    if missing:
        raise ValueError(f"ACTIVE_RECEIVER_BANDS tidak dikenali: {missing}")

    return {band: RECEIVER_BANDS[band] for band in ACTIVE_RECEIVER_BANDS}


def dataset_output_path_cnn_only(metadata, output_dir=OUTPUT_DIR):
    scenario_name = metadata["scenario_name"]
    scenario_type = metadata["scenario_type"]
    receiver_band = metadata["receiver_band"]
    return output_dir / f"{receiver_band}__{scenario_type}__{scenario_name}__cnn.npz"


def save_dataset_npz_cnn_only(dataset, metadata, output_dir=OUTPUT_DIR, overwrite=OVERWRITE_DATASETS):
    cnn_path = dataset_output_path_cnn_only(metadata, output_dir=output_dir)

    if not overwrite and cnn_path.exists():
        print("  SKIP SAVE: file sudah ada dan OVERWRITE_DATASETS=False")
        print("  CNN:", cnn_path)
        return cnn_path, True

    np.savez_compressed(
        cnn_path,
        X=dataset["cnn"]["X"],
        y=dataset["cnn"]["y"],
        metadata=json.dumps(metadata),
    )

    return cnn_path, False


def make_metadata_invariant(
    scenario_type,
    scenario_name,
    noise_var,
    fading_source,
    receiver_band,
    theta_ranges_2,
    path_loss_db,
):
    return {
        "scenario_type": scenario_type,
        "scenario_name": scenario_name,
        "noise_var": float(noise_var),
        "fading_source": fading_source,
        "path_loss_db": float(path_loss_db),
        "path_loss_linear_power_gain": float(path_loss_db_to_power_gain(path_loss_db)),
        "fading_std": float(math.sqrt(path_loss_db_to_power_gain(path_loss_db))),
        "receiver_band": receiver_band,
        "theta_ranges_1": [(float(a), float(b)) for a, b in theta_ranges_1],
        "theta_ranges_2": [(float(a), float(b)) for a, b in theta_ranges_2],
        **{k: (str(v) if k == "dtype" else v) for k, v in COMMON_CONFIG.items()},
    }


def generate_one_invariant_scenario_cnn_only(path_loss_db, receiver_band, theta_ranges_2):
    scenario_name = f"path_loss_minus_{abs(int(path_loss_db))}db"

    fading = make_invariant_fading_from_path_loss(
        path_loss_db=path_loss_db,
        total_antennas=COMMON_CONFIG["total_antennas"],
    )

    metadata = make_metadata_invariant(
        scenario_type=SCENARIO_TYPE_INVARIANT,
        scenario_name=scenario_name,
        noise_var=FIXED_NOISE_VAR_INVARIANT,
        fading_source="invariant_gaussian_fading_from_path_loss",
        receiver_band=receiver_band,
        theta_ranges_2=theta_ranges_2,
        path_loss_db=path_loss_db,
    )

    cnn_path = dataset_output_path_cnn_only(metadata)

    if not OVERWRITE_DATASETS and cnn_path.exists():
        print(f"\n=== SKIP {receiver_band} / {SCENARIO_TYPE_INVARIANT} / {scenario_name}: file sudah ada ===")
        return {
            "receiver_band": receiver_band,
            "scenario_type": SCENARIO_TYPE_INVARIANT,
            "scenario_name": scenario_name,
            "path_loss_db": float(path_loss_db),
            "cnn_path": str(cnn_path),
            "metadata": metadata,
            "status": "skipped_existing",
        }

    print(f"\n=== Membuat {receiver_band} / {SCENARIO_TYPE_INVARIANT} / {scenario_name} ===")
    print(f"  noise_var      : {FIXED_NOISE_VAR_INVARIANT}")
    print(f"  path_loss_db   : {path_loss_db}")
    print(f"  path_loss_linear : {path_loss_db_to_power_gain(path_loss_db):.6f}")
    print(f"  fading std       : {math.sqrt(path_loss_db_to_power_gain(path_loss_db)):.6f}")
    print(f"  fading vector  : {fading}")

    dataset = create_dataset_cnn_only(
        num_samples=COMMON_CONFIG["num_samples"],
        N1=COMMON_CONFIG["N1"],
        N2=COMMON_CONFIG["N2"],
        unfiltered_signal_length=COMMON_CONFIG["unfiltered_signal_length"],
        mean=COMMON_CONFIG["mean"],
        unfiltered_signal_var=COMMON_CONFIG["unfiltered_signal_var"],
        noise_var=FIXED_NOISE_VAR_INVARIANT,
        total_antennas=COMMON_CONFIG["total_antennas"],
        theta_ranges_1=theta_ranges_1,
        theta_ranges_2=theta_ranges_2,
        window_length=COMMON_CONFIG["window_length"],
        hop=COMMON_CONFIG["hop"],
        fading=fading,
        divide_matrix=COMMON_CONFIG["divide_matrix"],
        corrcoef=COMMON_CONFIG["corrcoef"],
        dtype=COMMON_CONFIG["dtype"],
    )

    cnn_path, skipped = save_dataset_npz_cnn_only(dataset, metadata)

    print("  CNN path :", cnn_path)
    print("  shape CNN:", dataset["cnn"]["X"].shape, dataset["cnn"]["y"].shape)
    print("  label 1  :", int(np.sum(dataset["cnn"]["y"] == 1)))
    print("  label 0  :", int(np.sum(dataset["cnn"]["y"] == 0)))

    return {
        "receiver_band": receiver_band,
        "scenario_type": SCENARIO_TYPE_INVARIANT,
        "scenario_name": scenario_name,
        "path_loss_db": float(path_loss_db),
        "cnn_path": str(cnn_path),
        "metadata": metadata,
        "status": "generated" if not skipped else "skipped_existing",
    }


def expected_dataset_count_invariant_cnn_only():
    receiver_count = len(selected_receiver_bands())
    condition_count = len(PATH_LOSS_DB_LIST)
    scenario_count = receiver_count * condition_count
    cnn_samples = scenario_count * COMMON_CONFIG["num_samples"] * 2

    return {
        "receiver_count": receiver_count,
        "condition_count_per_receiver": condition_count,
        "scenario_count": scenario_count,
        "cnn_samples": cnn_samples,
        "npz_files": scenario_count,
    }


def print_dataset_plan_invariant_cnn_only(minutes_per_scenario=2.5):
    info = expected_dataset_count_invariant_cnn_only()
    total_minutes = info["scenario_count"] * minutes_per_scenario

    print("Rencana dataset CNN-only:")
    print(f"  Receiver band / PU     : {info['receiver_count']}")
    print(f"  Path loss per PU       : {info['condition_count_per_receiver']}")
    print(f"  Total skenario dataset : {info['scenario_count']}")
    print(f"  Data CNN               : {info['cnn_samples']:,}".replace(",", "."))
    print(f"  Total file .npz        : {info['npz_files']}")
    print(f"  Estimasi generate      : {total_minutes:.1f} menit / {total_minutes/60:.2f} jam")

    return info


def generate_all_invariant_datasets_cnn_only():
    manifest = []

    for receiver_band, theta_ranges_2 in selected_receiver_bands().items():
        print("\n############################")
        print(f"# Receiver band: {receiver_band}")
        print("############################")

        for path_loss_db in PATH_LOSS_DB_LIST:
            manifest.append(
                generate_one_invariant_scenario_cnn_only(
                    path_loss_db=path_loss_db,
                    receiver_band=receiver_band,
                    theta_ranges_2=theta_ranges_2,
                )
            )

    manifest_path = OUTPUT_DIR / "manifest_invariant_cnn_only.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nManifest disimpan ke: {manifest_path}")
    print(f"Total entri manifest: {len(manifest)}")

    return manifest


### Jalankan pembuatan dataset invariant CNN-only


In [7]:
# Cek rencana jumlah dataset dan estimasi waktu.
print_dataset_plan_invariant_cnn_only(minutes_per_scenario=2.5)

# Dibuat True agar notebook bisa dijalankan sekali dari atas ke bawah.
# Jika runtime terputus, jalankan ulang saja. File yang sudah ada akan dilewati karena OVERWRITE_DATASETS=False.
RUN_DATASET_GENERATION = True

if RUN_DATASET_GENERATION:
    manifest = generate_all_invariant_datasets_cnn_only()
else:
    print("Dataset belum dibuat. Ubah RUN_DATASET_GENERATION=True untuk menjalankan generate_all_invariant_datasets_cnn_only().")


Rencana dataset CNN-only:
  Receiver band / PU     : 3
  Path loss per PU       : 4
  Total skenario dataset : 12
  Data CNN               : 24.000
  Total file .npz        : 12
  Estimasi generate      : 30.0 menit / 0.50 jam

############################
# Receiver band: pu1_low
############################

=== Membuat pu1_low / invariant_noise_statistics_and_fading / path_loss_minus_2db ===
  noise_var      : 80
  path_loss_db   : -2
  path_loss_linear : 0.630957
  fading std       : 0.794328
  fading vector  : [ 0.39455408 -0.10982724  0.51447729  1.20978562 -0.18599464 -0.1859816 ]
  sample 100/1000
  sample 200/1000
  sample 300/1000
  sample 400/1000
  sample 500/1000
  sample 600/1000
  sample 700/1000
  sample 800/1000
  sample 900/1000
  sample 1000/1000
  CNN path : dataset\generated_datasets\pu1_low__invariant_noise_statistics_and_fading__path_loss_minus_2db__cnn.npz
  shape CNN: (2000, 12, 12, 1) (2000,)
  label 1  : 1000
  label 0  : 1000

=== Membuat pu1_low / invariant

In [8]:
# # ============================================================
# # INSPEKSI SATU SAMPEL DATASET
# # Visualisasi:
# # 1. Domain waktu
# # 2. Domain frekuensi gaya lama: fftshift, -pi sampai pi
# # 3. PU + Noise warna ungu
# # 4. Noise-only warna merah
# # 5. Magnitude linear, bukan dB
# # 6. Matriks kovarian CNN dan CNN-LSTM
# # ============================================================

# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.fft import fft, fftshift


# # ============================================================
# # Konfigurasi warna
# # ============================================================

# COLOR_PU = "#090080"        # ungu
# COLOR_PU_NOISE = "#800080"  # ungu
# COLOR_NOISE = "#FF0000"     # merah


# # ============================================================
# # 1. Plot satu sinyal dalam domain waktu
# # ============================================================

# def plot_signal_time(
#     signal,
#     title="Domain Waktu",
#     max_points=None,
#     color="blue",
#     label=None,
# ):
#     signal = np.asarray(signal)

#     if max_points is not None:
#         signal = signal[:max_points]

#     plt.figure(figsize=(12, 4))
#     plt.plot(signal, color=color, label=label)

#     if label is not None:
#         plt.legend()

#     plt.title(title)
#     plt.xlabel("Sample index")
#     plt.ylabel("Amplitude")
#     plt.grid(True)
#     plt.show()


# # ============================================================
# # 2. Plot satu sinyal dalam domain frekuensi gaya lama
# #    Menggunakan fftshift dan magnitude linear, bukan dB
# # ============================================================

# def plot_signal_fft_old_style(
#     signal,
#     title="Domain Frekuensi",
#     max_points=None,
#     color="blue",
#     label=None,
# ):
#     signal = np.asarray(signal)

#     if max_points is not None:
#         signal = signal[:max_points]

#     N = len(signal)

#     # Sumbu frekuensi ternormalisasi:
#     # -1 sampai 1 merepresentasikan -pi sampai pi rad/sample
#     freq_axis = np.linspace(-1, 1, N, endpoint=False)

#     # FFT gaya lama: full spectrum + fftshift
#     # Magnitude linear, bukan dB
#     spectrum = fftshift(np.abs(fft(signal)))

#     plt.figure(figsize=(12, 4))
#     plt.plot(freq_axis, spectrum, color=color, label=label)

#     if label is not None:
#         plt.legend()

#     plt.title(title)
#     plt.xlabel("Normalized Frequency (×π rad/sample)")
#     plt.ylabel("Magnitude")
#     plt.grid(True)
#     plt.show()


# # ============================================================
# # 3. Plot beberapa window dalam domain waktu
# # ============================================================

# def plot_windows_time(
#     windows,
#     antenna_index=0,
#     max_windows=5,
#     color="blue",
#     label=None,
#     title_prefix="",
# ):
#     num_windows = min(len(windows), max_windows)

#     plt.figure(figsize=(12, 3 * num_windows))

#     for i in range(num_windows):
#         plt.subplot(num_windows, 1, i + 1)
#         plt.plot(windows[i][antenna_index], color=color, label=label)

#         if label is not None:
#             plt.legend()

#         plt.title(f"{title_prefix} - Domain Waktu - Window ke-{i} - Antena {antenna_index}")
#         plt.xlabel("Sample index dalam window")
#         plt.ylabel("Amplitude")
#         plt.grid(True)

#     plt.tight_layout()
#     plt.show()


# # ============================================================
# # 4. Plot beberapa window dalam domain frekuensi gaya lama
# # ============================================================

# def plot_windows_fft_old_style(
#     windows,
#     antenna_index=0,
#     max_windows=5,
#     color="blue",
#     label=None,
#     title_prefix="",
# ):
#     num_windows = min(len(windows), max_windows)

#     plt.figure(figsize=(12, 3 * num_windows))

#     for i in range(num_windows):
#         signal = np.asarray(windows[i][antenna_index])
#         N = len(signal)

#         freq_axis = np.linspace(-1, 1, N, endpoint=False)
#         spectrum = fftshift(np.abs(fft(signal)))

#         plt.subplot(num_windows, 1, i + 1)
#         plt.plot(freq_axis, spectrum, color=color, label=label)

#         if label is not None:
#             plt.legend()

#         plt.title(f"{title_prefix} - Domain Frekuensi - Window ke-{i} - Antena {antenna_index}")
#         plt.xlabel("Normalized Frequency (×π rad/sample)")
#         plt.ylabel("Magnitude")
#         plt.grid(True)

#     plt.tight_layout()
#     plt.show()


# # ============================================================
# # 5. Menggabungkan semua window berdasarkan antena tertentu
# # ============================================================

# def stack_windows_by_antenna(windows, antenna_index=0):
#     return np.hstack([window[antenna_index] for window in windows])


# # ============================================================
# # 6. Plot gabungan semua window dalam domain waktu dan frekuensi
# # ============================================================

# def plot_stacked_windows_time_and_fft(
#     windows,
#     antenna_index=0,
#     title_prefix="",
#     max_points_time=None,
#     max_points_fft=None,
#     color="blue",
#     label=None,
# ):
#     signal_full = stack_windows_by_antenna(
#         windows,
#         antenna_index=antenna_index
#     )

#     plot_signal_time(
#         signal_full,
#         title=f"{title_prefix} - Domain Waktu Gabungan Semua Window - Antena {antenna_index}",
#         max_points=max_points_time,
#         color=color,
#         label=label
#     )

#     plot_signal_fft_old_style(
#         signal_full,
#         title=f"{title_prefix} - Domain Frekuensi Gabungan Semua Window - Antena {antenna_index}",
#         max_points=max_points_fft,
#         color=color,
#         label=label
#     )

#     return signal_full


# # ============================================================
# # 7. Plot perbandingan PU + Noise dan Noise-only pada satu figure
# # ============================================================

# def plot_compare_time(
#     signal_pu_noise,
#     signal_noise,
#     title="Perbandingan Domain Waktu",
#     max_points=None,
# ):
#     signal_pu_noise = np.asarray(signal_pu_noise)
#     signal_noise = np.asarray(signal_noise)

#     if max_points is not None:
#         signal_pu_noise = signal_pu_noise[:max_points]
#         signal_noise = signal_noise[:max_points]

#     plt.figure(figsize=(12, 4))
#     plt.plot(signal_pu_noise, color=COLOR_PU_NOISE, label="PU + Noise", alpha=0.85)
#     plt.plot(signal_noise, color=COLOR_NOISE, label="Noise-only", alpha=0.75)
#     plt.title(title)
#     plt.xlabel("Sample index")
#     plt.ylabel("Amplitude")
#     plt.legend()
#     plt.grid(True)
#     plt.show()


# def plot_compare_fft_old_style(
#     signal_pu_noise,
#     signal_noise,
#     title="Perbandingan Domain Frekuensi",
#     max_points=None,
# ):
#     signal_pu_noise = np.asarray(signal_pu_noise)
#     signal_noise = np.asarray(signal_noise)

#     if max_points is not None:
#         signal_pu_noise = signal_pu_noise[:max_points]
#         signal_noise = signal_noise[:max_points]

#     N_pu = len(signal_pu_noise)
#     N_noise = len(signal_noise)

#     min_N = min(N_pu, N_noise)
#     signal_pu_noise = signal_pu_noise[:min_N]
#     signal_noise = signal_noise[:min_N]

#     freq_axis = np.linspace(-1, 1, min_N, endpoint=False)

#     spectrum_pu_noise = fftshift(np.abs(fft(signal_pu_noise)))
#     spectrum_noise = fftshift(np.abs(fft(signal_noise)))

#     plt.figure(figsize=(12, 4))
#     plt.plot(freq_axis, spectrum_pu_noise, color=COLOR_PU_NOISE, label="PU + Noise", alpha=0.85)
#     plt.plot(freq_axis, spectrum_noise, color=COLOR_NOISE, label="Noise-only", alpha=0.75)
#     plt.title(title)
#     plt.xlabel("Normalized Frequency (×π rad/sample)")
#     plt.ylabel("Magnitude")
#     plt.legend()
#     plt.grid(True)
#     plt.show()


# # ============================================================
# # 8. Plot matriks kovarian
# # ============================================================

# def plot_matrix(matrix, title="Matriks Kovarian"):
#     plt.figure(figsize=(6, 5))
#     plt.imshow(matrix, aspect="auto")
#     plt.colorbar()
#     plt.title(title)
#     plt.xlabel("Index")
#     plt.ylabel("Index")
#     plt.show()


# # ============================================================
# # 9. Fungsi utama inspeksi satu sampel
# # ============================================================

# def inspect_one_sample_complete_visual(
#     N1,
#     N2,
#     unfiltered_signal_length,
#     mean,
#     unfiltered_signal_var,
#     noise_var,
#     total_antennas,
#     theta_ranges_1,
#     theta_ranges_2,
#     window_length,
#     hop,
#     fading,
#     divide_matrix=True,
#     corrcoef=False,
#     antenna_index=0,
#     max_windows=5,
#     max_points_time=None,
#     max_points_fft=None,
#     show_per_window=True,
#     show_stacked=True,
#     show_compare=True,
#     show_covariance=True,
# ):
#     print("============================================================")
#     print("1. MEMBUAT ISYARAT PU SEBELUM WINDOWING")
#     print("============================================================")

#     PU_Signal = generate_pu_signal(
#         theta_ranges_1,
#         mean,
#         unfiltered_signal_var,
#         unfiltered_signal_length,
#         N1,
#         total_antennas,
#     )

#     print("PU_Signal shape:", PU_Signal.shape)

#     plot_signal_time(
#         PU_Signal[antenna_index],
#         title=f"PU sebelum Windowing - Domain Waktu - Antena {antenna_index}",
#         max_points=max_points_time,
#         color=COLOR_PU,
#         label="PU"
#     )

#     plot_signal_fft_old_style(
#         PU_Signal[antenna_index],
#         title=f"PU sebelum Windowing - Domain Frekuensi - Antena {antenna_index}",
#         max_points=max_points_fft,
#         color=COLOR_PU,
#         label="PU"
#     )


#     print("============================================================")
#     print("2. MEMBUAT WINDOW PU + NOISE DAN NOISE-ONLY")
#     print("============================================================")

#     shared_windows_pu, shared_windows_noise = generate_windowed_data(
#         PU_Signal,
#         window_length,
#         hop,
#         mean,
#         noise_var,
#         fading,
#     )

#     print("Jumlah window:", len(shared_windows_pu))
#     print("Shape window PU + noise pertama:", shared_windows_pu[0].shape)
#     print("Shape window noise-only pertama:", shared_windows_noise[0].shape)


#     if show_per_window:
#         print("============================================================")
#         print("2a. PU + NOISE PER WINDOW - DOMAIN WAKTU")
#         print("============================================================")

#         plot_windows_time(
#             shared_windows_pu,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise",
#             title_prefix="PU + Noise sebelum Filtering"
#         )

#         print("============================================================")
#         print("2b. PU + NOISE PER WINDOW - DOMAIN FREKUENSI")
#         print("============================================================")

#         plot_windows_fft_old_style(
#             shared_windows_pu,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise",
#             title_prefix="PU + Noise sebelum Filtering"
#         )

#         print("============================================================")
#         print("2c. NOISE-ONLY PER WINDOW - DOMAIN WAKTU")
#         print("============================================================")

#         plot_windows_time(
#             shared_windows_noise,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_NOISE,
#             label="Noise-only",
#             title_prefix="Noise-only sebelum Filtering"
#         )

#         print("============================================================")
#         print("2d. NOISE-ONLY PER WINDOW - DOMAIN FREKUENSI")
#         print("============================================================")

#         plot_windows_fft_old_style(
#             shared_windows_noise,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_NOISE,
#             label="Noise-only",
#             title_prefix="Noise-only sebelum Filtering"
#         )


#     print("============================================================")
#     print("3. MEMBUAT FILTER RECEIVER")
#     print("============================================================")

#     H = build_receiver_filter(theta_ranges_2, N2, window_length)

#     print("H shape:", H.shape)


#     print("============================================================")
#     print("4. FILTERING RECEIVER PER WINDOW")
#     print("============================================================")

#     windows_filtered_pu = filter_windows(shared_windows_pu, H)
#     windows_filtered_noise = filter_windows(shared_windows_noise, H)

#     print("Shape window PU + noise setelah filtering pertama:", windows_filtered_pu[0].shape)
#     print("Shape window noise-only setelah filtering pertama:", windows_filtered_noise[0].shape)


#     if show_per_window:
#         print("============================================================")
#         print("4a. PU + NOISE SETELAH FILTERING PER WINDOW - DOMAIN WAKTU")
#         print("============================================================")

#         plot_windows_time(
#             windows_filtered_pu,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise",
#             title_prefix="PU + Noise setelah Filtering"
#         )

#         print("============================================================")
#         print("4b. PU + NOISE SETELAH FILTERING PER WINDOW - DOMAIN FREKUENSI")
#         print("============================================================")

#         plot_windows_fft_old_style(
#             windows_filtered_pu,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise",
#             title_prefix="PU + Noise setelah Filtering"
#         )

#         print("============================================================")
#         print("4c. NOISE-ONLY SETELAH FILTERING PER WINDOW - DOMAIN WAKTU")
#         print("============================================================")

#         plot_windows_time(
#             windows_filtered_noise,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_NOISE,
#             label="Noise-only",
#             title_prefix="Noise-only setelah Filtering"
#         )

#         print("============================================================")
#         print("4d. NOISE-ONLY SETELAH FILTERING PER WINDOW - DOMAIN FREKUENSI")
#         print("============================================================")

#         plot_windows_fft_old_style(
#             windows_filtered_noise,
#             antenna_index=antenna_index,
#             max_windows=max_windows,
#             color=COLOR_NOISE,
#             label="Noise-only",
#             title_prefix="Noise-only setelah Filtering"
#         )


#     print("============================================================")
#     print("5. MENGGABUNGKAN SEMUA WINDOW UNTUK VISUALISASI")
#     print("============================================================")

#     signal_pu_noise_stacked = stack_windows_by_antenna(
#         shared_windows_pu,
#         antenna_index=antenna_index
#     )

#     signal_noise_stacked = stack_windows_by_antenna(
#         shared_windows_noise,
#         antenna_index=antenna_index
#     )

#     signal_pu_noise_filtered_stacked = stack_windows_by_antenna(
#         windows_filtered_pu,
#         antenna_index=antenna_index
#     )

#     signal_noise_filtered_stacked = stack_windows_by_antenna(
#         windows_filtered_noise,
#         antenna_index=antenna_index
#     )

#     print("Shape PU + noise gabungan sebelum filtering:", signal_pu_noise_stacked.shape)
#     print("Shape noise-only gabungan sebelum filtering:", signal_noise_stacked.shape)
#     print("Shape PU + noise gabungan setelah filtering:", signal_pu_noise_filtered_stacked.shape)
#     print("Shape noise-only gabungan setelah filtering:", signal_noise_filtered_stacked.shape)


#     if show_stacked:
#         print("============================================================")
#         print("5a. PU + NOISE GABUNGAN SEBELUM FILTERING")
#         print("============================================================")

#         plot_signal_time(
#             signal_pu_noise_stacked,
#             title=f"PU + Noise sebelum Filtering - Domain Waktu Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_time,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise"
#         )

#         plot_signal_fft_old_style(
#             signal_pu_noise_stacked,
#             title=f"PU + Noise sebelum Filtering - Domain Frekuensi Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_fft,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise"
#         )


#         print("============================================================")
#         print("5b. NOISE-ONLY GABUNGAN SEBELUM FILTERING")
#         print("============================================================")

#         plot_signal_time(
#             signal_noise_stacked,
#             title=f"Noise-only sebelum Filtering - Domain Waktu Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_time,
#             color=COLOR_NOISE,
#             label="Noise-only"
#         )

#         plot_signal_fft_old_style(
#             signal_noise_stacked,
#             title=f"Noise-only sebelum Filtering - Domain Frekuensi Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_fft,
#             color=COLOR_NOISE,
#             label="Noise-only"
#         )


#         print("============================================================")
#         print("5c. PU + NOISE GABUNGAN SETELAH FILTERING")
#         print("============================================================")

#         plot_signal_time(
#             signal_pu_noise_filtered_stacked,
#             title=f"PU + Noise setelah Filtering - Domain Waktu Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_time,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise"
#         )

#         plot_signal_fft_old_style(
#             signal_pu_noise_filtered_stacked,
#             title=f"PU + Noise setelah Filtering - Domain Frekuensi Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_fft,
#             color=COLOR_PU_NOISE,
#             label="PU + Noise"
#         )


#         print("============================================================")
#         print("5d. NOISE-ONLY GABUNGAN SETELAH FILTERING")
#         print("============================================================")

#         plot_signal_time(
#             signal_noise_filtered_stacked,
#             title=f"Noise-only setelah Filtering - Domain Waktu Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_time,
#             color=COLOR_NOISE,
#             label="Noise-only"
#         )

#         plot_signal_fft_old_style(
#             signal_noise_filtered_stacked,
#             title=f"Noise-only setelah Filtering - Domain Frekuensi Gabungan Semua Window - Antena {antenna_index}",
#             max_points=max_points_fft,
#             color=COLOR_NOISE,
#             label="Noise-only"
#         )


#     if show_compare:
#         print("============================================================")
#         print("6. PERBANDINGAN PU + NOISE DAN NOISE-ONLY DALAM SATU FIGURE")
#         print("============================================================")

#         plot_compare_time(
#             signal_pu_noise_stacked,
#             signal_noise_stacked,
#             title=f"Perbandingan Sebelum Filtering - Domain Waktu - Antena {antenna_index}",
#             max_points=max_points_time
#         )

#         plot_compare_fft_old_style(
#             signal_pu_noise_stacked,
#             signal_noise_stacked,
#             title=f"Perbandingan Sebelum Filtering - Domain Frekuensi - Antena {antenna_index}",
#             max_points=max_points_fft
#         )

#         plot_compare_time(
#             signal_pu_noise_filtered_stacked,
#             signal_noise_filtered_stacked,
#             title=f"Perbandingan Setelah Filtering - Domain Waktu - Antena {antenna_index}",
#             max_points=max_points_time
#         )

#         plot_compare_fft_old_style(
#             signal_pu_noise_filtered_stacked,
#             signal_noise_filtered_stacked,
#             title=f"Perbandingan Setelah Filtering - Domain Frekuensi - Antena {antenna_index}",
#             max_points=max_points_fft
#         )


#     print("============================================================")
#     print("7. MEMBUAT MATRIKS KOVARIAN UNTUK CNN")
#     print("============================================================")

#     corr_pu_cnn = make_cnn_matrix(
#         shared_windows_pu,
#         H,
#         divide_matrix=divide_matrix,
#         corrcoef=corrcoef
#     )

#     corr_noise_cnn = make_cnn_matrix(
#         shared_windows_noise,
#         H,
#         divide_matrix=divide_matrix,
#         corrcoef=corrcoef
#     )

#     print("Shape matriks kovarian PU + noise CNN:", corr_pu_cnn.shape)
#     print("Shape matriks kovarian noise-only CNN:", corr_noise_cnn.shape)


#     print("============================================================")
#     print("8. MEMBUAT SEQUENCE MATRIKS KOVARIAN UNTUK CNN-LSTM")
#     print("============================================================")

#     corr_pu_cnn_lstm = make_cnn_lstm_sequence(
#         shared_windows_pu,
#         H,
#         divide_matrix=divide_matrix,
#         corrcoef=corrcoef
#     )

#     corr_noise_cnn_lstm = make_cnn_lstm_sequence(
#         shared_windows_noise,
#         H,
#         divide_matrix=divide_matrix,
#         corrcoef=corrcoef
#     )

#     print("Shape sequence kovarian PU + noise CNN-LSTM:", corr_pu_cnn_lstm.shape)
#     print("Shape sequence kovarian noise-only CNN-LSTM:", corr_noise_cnn_lstm.shape)


#     if show_covariance:
#         print("============================================================")
#         print("9. VISUALISASI MATRIKS KOVARIAN")
#         print("============================================================")

#         plot_matrix(
#             corr_pu_cnn,
#             title="Matriks Kovarian PU + Noise untuk CNN"
#         )

#         plot_matrix(
#             corr_noise_cnn,
#             title="Matriks Kovarian Noise-only untuk CNN"
#         )

#         plot_matrix(
#             corr_pu_cnn_lstm[0],
#             title="Matriks Kovarian PU + Noise CNN-LSTM pada Window ke-0"
#         )

#         plot_matrix(
#             corr_noise_cnn_lstm[0],
#             title="Matriks Kovarian Noise-only CNN-LSTM pada Window ke-0"
#         )


#     return {
#         "PU_Signal": PU_Signal,

#         "shared_windows_pu": shared_windows_pu,
#         "shared_windows_noise": shared_windows_noise,

#         "windows_filtered_pu": windows_filtered_pu,
#         "windows_filtered_noise": windows_filtered_noise,

#         "signal_pu_noise_stacked": signal_pu_noise_stacked,
#         "signal_noise_stacked": signal_noise_stacked,

#         "signal_pu_noise_filtered_stacked": signal_pu_noise_filtered_stacked,
#         "signal_noise_filtered_stacked": signal_noise_filtered_stacked,

#         "corr_pu_cnn": corr_pu_cnn,
#         "corr_noise_cnn": corr_noise_cnn,

#         "corr_pu_cnn_lstm": corr_pu_cnn_lstm,
#         "corr_noise_cnn_lstm": corr_noise_cnn_lstm,

#         "H": H,
#     }

In [9]:
# theta_ranges_1 = [
#     (np.pi * 0.1, np.pi * 0.25),  # PU 1
#     (np.pi * 0.4, np.pi * 0.55),  # PU 2
#     (np.pi * 0.7, np.pi * 0.85),  # PU 3
# ]

# # Tiga band penerima/detektor. Dataset akan dibuat untuk semuanya.
# theta_ranges_2 = {
#     (np.pi * 0.08, np.pi * 0.252),  # PU 1
#     # (np.pi * 0.4, np.pi * 0.55),  # PU 2
#     # (np.pi * 0.7, np.pi * 0.85),  # PU 3
# }

# fading = load_fading_csv("dataset/fading-10.csv")

# debug_result = inspect_one_sample_complete_visual(
#     N1=COMMON_CONFIG["N1"],
#     N2=COMMON_CONFIG["N2"],
#     unfiltered_signal_length=COMMON_CONFIG["unfiltered_signal_length"],
#     mean=COMMON_CONFIG["mean"],
#     unfiltered_signal_var=COMMON_CONFIG["unfiltered_signal_var"],
#     noise_var=80,
#     total_antennas=COMMON_CONFIG["total_antennas"],
#     theta_ranges_1=theta_ranges_1,
#     theta_ranges_2=theta_ranges_2,
#     window_length=COMMON_CONFIG["window_length"],
#     hop=COMMON_CONFIG["hop"],
#     fading=fading,
#     divide_matrix=True,
#     corrcoef=False,
#     antenna_index=0,
# )

## 7. Fungsi Load Dataset CNN-only


In [10]:
def load_npz_dataset(path):
    data = np.load(path, allow_pickle=False)
    X = data["X"]
    y = data["y"]
    metadata = json.loads(data["metadata"].item())
    return X, y, metadata


def load_manifest(manifest_path=OUTPUT_DIR / "manifest_invariant_cnn_only.json"):
    with open(manifest_path, "r", encoding="utf-8") as f:
        return json.load(f)


def select_dataset_files(manifest, model_type, scenario_types=None, scenario_names=None, receiver_bands=None):
    if model_type != "cnn":
        raise ValueError("Notebook ini disiapkan untuk CNN-only. model_type harus 'cnn'.")

    selected = []
    for item in manifest:
        if scenario_types is not None and item["scenario_type"] not in scenario_types:
            continue
        if scenario_names is not None and item["scenario_name"] not in scenario_names:
            continue
        if receiver_bands is not None and item["receiver_band"] not in receiver_bands:
            continue
        selected.append(item["cnn_path"])

    if not selected:
        raise ValueError("Tidak ada file dataset yang cocok dengan filter yang diberikan.")

    return selected


def load_and_concat(paths):
    X_list, y_list, metadata_list = [], [], []
    for path in paths:
        X, y, metadata = load_npz_dataset(path)
        X_list.append(X)
        y_list.append(y)
        metadata_list.append(metadata)

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.concatenate(y_list, axis=0)
    return X_all, y_all, metadata_list


## 7.1 Validasi Manifest dan Dataset CNN-only


In [11]:
def validate_manifest_and_files(manifest):
    expected = expected_dataset_count_invariant_cnn_only()
    print("Jumlah entri manifest:", len(manifest))
    print("Ekspektasi skenario dataset:", expected["scenario_count"])

    missing = []
    cnn_samples = 0

    for item in manifest:
        cnn_path = Path(item["cnn_path"])

        if not cnn_path.exists():
            missing.append(str(cnn_path))
        else:
            with np.load(cnn_path, allow_pickle=False) as data:
                cnn_samples += data["X"].shape[0]

    print("Data CNN ditemukan:", f"{cnn_samples:,}".replace(',', '.'))
    print("Ekspektasi data CNN:", f"{expected['cnn_samples']:,}".replace(',', '.'))

    if missing:
        print("File hilang:")
        for path in missing[:20]:
            print(" -", path)
        if len(missing) > 20:
            print(f"... dan {len(missing) - 20} file lain")
    else:
        print("Semua file CNN pada manifest ditemukan.")

    return {
        "manifest_entries": len(manifest),
        "cnn_samples": cnn_samples,
        "missing_files": missing,
    }


# Validasi otomatis setelah dataset dibuat atau manifest tersedia.
if 'manifest' not in globals():
    manifest = load_manifest()

validation = validate_manifest_and_files(manifest)


Jumlah entri manifest: 12
Ekspektasi skenario dataset: 12
Data CNN ditemukan: 24.000
Ekspektasi data CNN: 24.000
Semua file CNN pada manifest ditemukan.


## 8. Model CNN dan CNN-LSTM untuk Grid Search

Fungsi model dibuat parametrik agar dapat dipakai oleh grid search. Grid search di sini bukan untuk mengubah metodologi penelitian utama, melainkan untuk mencari kombinasi parameter pelatihan dan arsitektur yang paling stabil pada dataset gabungan.


In [12]:
def make_optimizer(name, learning_rate):
    name = name.lower()
    if name == "adam":
        return optimizers.Adam(learning_rate=learning_rate)
    if name == "sgd":
        return optimizers.SGD(learning_rate=learning_rate)
    if name == "rmsprop":
        return optimizers.RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Optimizer tidak dikenali: {name}")


def build_cnn_model(input_shape, filters=(32, 64), kernel_size=3, dense_units=128, pooling="max", dropout=0.0,
                    optimizer_name="adam", learning_rate=1e-3):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    for f in filters:
        x = layers.Conv2D(f, (kernel_size, kernel_size), activation="relu", padding="same")(x)
        x = layers.MaxPooling2D((2, 2))(x)

    if pooling == "gap":
        x = layers.GlobalAveragePooling2D()(x)
    elif pooling == "flatten":
        x = layers.Flatten()(x)
    else:
        raise ValueError("pooling harus 'gap' atau 'flatten'")

    x = layers.Dense(dense_units, activation="relu")(x)
    # if dropout > 0:
    #     x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(2, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_cnn_encoder(input_shape, encoder_filters=(16, 32), kernel_size=3, encoder_pooling="gap"):
    inputs = layers.Input(shape=input_shape)
    x = inputs

    for f in encoder_filters:
        x = layers.Conv2D(f, (kernel_size, kernel_size), activation="relu", padding="same", strides=2)(x)

    if encoder_pooling == "gap":
        x = layers.GlobalAveragePooling2D()(x)
    elif encoder_pooling == "flatten":
        x = layers.Flatten()(x)
    else:
        raise ValueError("encoder_pooling harus 'gap' atau 'flatten'")

    return models.Model(inputs, x)


def build_cnn_lstm_model(input_shape, encoder_filters=(16, 32), kernel_size=3, lstm_units=32, dense_units=0,
                         encoder_pooling="gap", dropout=0.0, optimizer_name="adam", learning_rate=1e-3):
    # input_shape: (time_steps, height, width, channels)
    time_steps = input_shape[0]
    frame_shape = input_shape[1:]

    encoder = build_cnn_encoder(
        input_shape=frame_shape,
        encoder_filters=encoder_filters,
        kernel_size=kernel_size,
        encoder_pooling=encoder_pooling,
    )

    inputs = layers.Input(shape=input_shape)
    x = layers.TimeDistributed(encoder)(inputs)
    x = layers.LSTM(lstm_units, return_sequences=False)(x)

    if dense_units and dense_units > 0:
        x = layers.Dense(dense_units, activation="relu")(x)

    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(2, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


## 9. Manual Grid Search

Grid search dibuat manual agar tidak bergantung pada library tambahan seperti SciKeras atau Keras Tuner. Metrik utama menggunakan `val_accuracy`. Setelah kombinasi terbaik ditemukan, model terbaik dievaluasi pada test set.

Pembagian data default pada fungsi `manual_grid_search()` adalah:

```text
Train      = 70%
Validation = 10%
Test       = 20%
```

Pembagian dilakukan secara stratified sehingga proporsi label 1 (PU + noise) dan label 0 (noise only) tetap seimbang pada train, validation, dan test.


In [13]:
def detection_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    pd_value = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pfa_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    youden_index = pd_value - pfa_value
    return pd_value, pfa_value, youden_index


def expand_param_grid(param_grid):
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]
    for combo in product(*values):
        yield dict(zip(keys, combo))


def count_param_combinations(param_grid):
    total = 1
    for values in param_grid.values():
        total *= len(values)
    return total


def print_grid_search_plan(minutes_per_training=2.0):
    cnn_runs = count_param_combinations(CNN_PARAM_GRID)
    total_minutes = cnn_runs * minutes_per_training
    print("Rencana grid search dataset invariant CNN-only:")
    print(f"  CNN      : {cnn_runs} training")
    print(f"  Total    : {cnn_runs} training")
    print(f"  Estimasi : {total_minutes:.1f} menit / {total_minutes/60:.2f} jam")
    return {"cnn_runs": cnn_runs, "total_runs": cnn_runs, "total_minutes": total_minutes}


def make_global_split_indices(y, train_size=0.7, val_size=0.1, test_size=0.2, random_state=42):
    """
    Membuat indeks train/validation/test dari dataset gabungan besar.

    Catatan penting:
    - Split dilakukan pada indeks global, bukan langsung pada X.
    - Indeks ini kemudian dipakai ulang untuk:
      1) training grid search,
      2) evaluasi test global,
      3) inference/metrik per skenario.
    Dengan demikian, data test per skenario berasal dari test set yang sama persis
    dengan test set yang dipakai saat evaluasi model setelah training.
    """
    total_split = train_size + val_size + test_size
    if not np.isclose(total_split, 1.0):
        raise ValueError(f"train_size + val_size + test_size harus = 1.0, tetapi sekarang = {total_split}")

    indices = np.arange(len(y))

    idx_train_val, idx_test, y_train_val, y_test = train_test_split(
        indices,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    val_size_relative = val_size / (train_size + val_size)
    idx_train, idx_val, y_train, y_val = train_test_split(
        idx_train_val,
        y_train_val,
        test_size=val_size_relative,
        stratify=y_train_val,
        random_state=random_state,
    )

    # print("idx_train[:100] =", idx_train[:100])
    # print("idx_val[:100]   =", idx_val[:100])
    # print("idx_test[:100]  =", idx_test[:100])

    split_indices = {
        "idx_train": idx_train,
        "idx_val": idx_val,
        "idx_test": idx_test,
    }

    print("Pembagian data:")
    print(f"  Train      : {len(idx_train):,} data ({len(idx_train) / len(y) * 100:.1f}%)")
    print(f"  Validation : {len(idx_val):,} data ({len(idx_val) / len(y) * 100:.1f}%)")
    print(f"  Test       : {len(idx_test):,} data ({len(idx_test) / len(y) * 100:.1f}%)")

    return split_indices


def save_global_split_indices(split_indices, model_type, output_dir=OUTPUT_DIR):
    split_path = output_dir / f"global_split_indices_{model_type}.npz"
    np.savez_compressed(
        split_path,
        idx_train=split_indices["idx_train"],
        idx_val=split_indices["idx_val"],
        idx_test=split_indices["idx_test"],
        seed=SEED,
        train_size=0.7,
        val_size=0.1,
        test_size=0.2,
    )
    print(f"Split indices {model_type} disimpan ke:", split_path)
    return split_path


def manual_grid_search(X, y, model_type, param_grid, train_size=0.7, val_size=0.1, test_size=0.2, random_state=42):
    # Split dibuat sekali, lalu dipakai untuk semua kombinasi hyperparameter.
    split_indices = make_global_split_indices(
        y,
        train_size=train_size,
        val_size=val_size,
        test_size=test_size,
        random_state=random_state,
    )

    idx_train = split_indices["idx_train"]
    idx_val = split_indices["idx_val"]
    idx_test = split_indices["idx_test"]

    X_train, y_train = X[idx_train], y[idx_train]
    X_val, y_val = X[idx_val], y[idx_val]
    X_test, y_test = X[idx_test], y[idx_test]

    # Simpan indeks agar inference/metrik per skenario tidak perlu merekonstruksi split.
    save_global_split_indices(split_indices, model_type=model_type)

    results = []
    best = None
    param_combinations = list(expand_param_grid(param_grid))

    for run_index, raw_params in enumerate(param_combinations, start=1):
        params = dict(raw_params)
        print(f"\n=== Grid run {run_index}/{len(param_combinations)} | {model_type} ===")
        print(params)

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)

        epochs = params.pop("epochs")
        batch_size = params.pop("batch_size")

        fit_kwargs = {
            "epochs": epochs,
            "batch_size": batch_size,
            "verbose": 1,
            "callbacks": [
                callbacks.EarlyStopping(
                    monitor="val_accuracy",
                    patience=5,
                    mode="max",
                    restore_best_weights=True,
                )
            ],
        }

        if model_type == "cnn":
            model = build_cnn_model(input_shape=X.shape[1:], **params)
        elif model_type == "cnn_lstm":
            model = build_cnn_lstm_model(input_shape=X.shape[1:], **params)
        else:
            raise ValueError("model_type harus 'cnn' atau 'cnn_lstm'")

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            **fit_kwargs,
        )

        val_acc = max(history.history["val_accuracy"])
        train_acc = max(history.history["accuracy"])
        best_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)

        row = {
            "run_index": run_index,
            "model_type": model_type,
            "val_accuracy": float(val_acc),
            "train_accuracy": float(train_acc),
            "best_epoch": best_epoch,
            **raw_params,
        }
        results.append(row)

        if best is None or val_acc > best["val_accuracy"]:
            best = {
                "val_accuracy": float(val_acc),
                "model": model,
                "params": raw_params,
                "history": history,
                "split_indices": split_indices,
                "best_epoch": best_epoch,
            }

    results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
    return results_df, best


def evaluate_best_model(best, X, y):
    model = best["model"]
    idx_test = best["split_indices"]["idx_test"]

    X_test = X[idx_test]
    y_test = y[idx_test]

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    y_prob_raw = model.predict(X_test, verbose=0)

    if y_prob_raw.ndim == 2 and y_prob_raw.shape[1] == 2:
        y_prob = y_prob_raw[:, 1]
        y_pred = np.argmax(y_prob_raw, axis=1)
    elif y_prob_raw.ndim == 2 and y_prob_raw.shape[1] == 1:
        y_prob = y_prob_raw[:, 0]
        y_pred = (y_prob >= 0.5).astype(int)
    elif y_prob_raw.ndim == 1:
        y_prob = y_prob_raw
        y_pred = (y_prob >= 0.5).astype(int)
    else:
        raise ValueError(f"Bentuk output model tidak dikenali: {y_prob_raw.shape}")

    pd_value, pfa_value, youden_index = detection_metrics(y_test, y_pred)

    print("Test loss:", test_loss)
    print("Test accuracy:", test_acc)
    print("Pd:", pd_value)
    print("Pfa:", pfa_value)
    print("Youden Index:", youden_index)
    print("Best params:", best["params"])
    print("Best epoch:", best["best_epoch"])
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

    return {
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "pd": float(pd_value),
        "pfa": float(pfa_value),
        "youden_index": float(youden_index),
        "y_pred": y_pred,
        "y_prob": y_prob,
        "best_params": best["params"],
        "best_epoch": best["best_epoch"],
        "test_size": int(len(y_test)),
    }


## 9.1 Evaluasi Test Per Skenario dengan Split yang Sama dari Training

Bagian ini menggabungkan logika inference ke notebook training. Metrik per skenario dihitung memakai `idx_test` yang sama persis dengan split yang dipakai saat grid search.


In [14]:
def build_dataset_table(manifest, model_type):
    """
    Mengubah manifest menjadi tabel dataset per skenario untuk model tertentu.
    Tabel ini menjaga urutan file persis seperti proses load dataset gabungan.
    """
    if model_type != "cnn":
        raise ValueError("Notebook ini disiapkan untuk CNN-only. model_type harus 'cnn'.")

    rows = []
    path_key = "cnn_path"

    for item_index, item in enumerate(manifest):
        rows.append({
            "item_index": item_index,
            "model_type": model_type,
            "receiver_band": item["receiver_band"],
            "scenario_type": item["scenario_type"],
            "scenario_name": item["scenario_name"],
            "dataset_path": item[path_key],
        })

    return pd.DataFrame(rows)


def collect_dataset_ranges_from_table(dataset_table):
    """
    Membuat peta indeks global untuk setiap file skenario.

    Contoh:
    file skenario pertama  : indeks global 0-1999
    file skenario kedua    : indeks global 2000-3999
    dst.
    """
    ranges = []
    start = 0

    for _, row in dataset_table.iterrows():
        _, y, _ = load_npz_dataset(row["dataset_path"])
        end = start + len(y)
        ranges.append({
            "item_index": int(row["item_index"]),
            "start": start,
            "end": end,
            "n_total": len(y),
            "dataset_path": row["dataset_path"],
        })
        start = end

    return pd.DataFrame(ranges)


def local_test_indices_for_file(global_test_indices, file_start, file_end):
    """
    Mengambil indeks test global yang berada pada rentang file tertentu,
    lalu mengubahnya menjadi indeks lokal file tersebut.
    """
    global_test_indices = np.asarray(global_test_indices)
    mask = (global_test_indices >= file_start) & (global_test_indices < file_end)
    return global_test_indices[mask] - file_start


def predict_binary(model, X, threshold=0.5, batch_size=256):
    raw = model.predict(X, batch_size=batch_size, verbose=0)

    # Model yang dipakai di notebook ini memakai Dense(2, activation='softmax').
    # Jika suatu saat diganti menjadi sigmoid 1 output, fungsi ini tetap aman.
    if raw.ndim == 2 and raw.shape[1] == 2:
        proba = raw[:, 1]
    elif raw.ndim == 2 and raw.shape[1] == 1:
        proba = raw[:, 0]
    elif raw.ndim == 1:
        proba = raw
    else:
        raise ValueError(f"Bentuk output model tidak dikenali: {raw.shape}")

    y_pred = (proba >= threshold).astype(int)
    return y_pred, proba


def compute_binary_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    pd_value = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pfa_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    youden_index = pd_value - pfa_value

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "pd": pd_value,
        "pfa": pfa_value,
        "youden_index": youden_index,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support_0_noise_only": int(tn + fp),
        "support_1_pu_noise": int(tp + fn),
    }


def evaluate_one_dataset_file_from_training_split(
    model,
    row,
    ranges_df,
    split_indices,
    threshold=0.5,
):
    """
    Evaluasi satu file skenario memakai idx_test yang sama persis dari training.
    """
    X, y, metadata = load_npz_dataset(row["dataset_path"])

    range_row = ranges_df.loc[ranges_df["item_index"] == int(row["item_index"])].iloc[0]
    eval_indices = local_test_indices_for_file(
        split_indices["idx_test"],
        int(range_row["start"]),
        int(range_row["end"]),
    )

    if len(eval_indices) == 0:
        raise ValueError(f"Tidak ada data test untuk file: {row['dataset_path']}")

    X_eval = X[eval_indices]
    y_eval = y[eval_indices]
    y_pred, proba = predict_binary(model, X_eval, threshold=threshold)
    metrics = compute_binary_metrics(y_eval, y_pred)

    result = {
        "model_type": row["model_type"],
        "receiver_band": row["receiver_band"],
        "scenario_type": row["scenario_type"],
        "scenario_name": row["scenario_name"],
        "evaluation_mode": "training_test_split",
        "threshold": threshold,
        "n_total_file": int(len(y)),
        "n_eval": int(len(y_eval)),
        "dataset_path": row["dataset_path"],
    }
    result.update(metrics)
    return result


def evaluate_model_per_scenario_from_training_split(
    model,
    dataset_table,
    ranges_df,
    split_indices,
    threshold=0.5,
):
    rows = []

    for i, (_, row) in enumerate(dataset_table.iterrows(), start=1):
        print(f"[{i}/{len(dataset_table)}] {row['model_type']} | {row['receiver_band']} | {row['scenario_type']} | {row['scenario_name']}")
        result = evaluate_one_dataset_file_from_training_split(
            model=model,
            row=row,
            ranges_df=ranges_df,
            split_indices=split_indices,
            threshold=threshold,
        )
        rows.append(result)

    return pd.DataFrame(rows)


def export_inference_results(results_df, evaluation_mode="training_test_split", output_dir=OUTPUT_DIR):
    if results_df is None or results_df.empty:
        print("Tidak ada hasil inference untuk disimpan.")
        return None

    inference_csv_path = output_dir / f"inference_metrics_per_scenario__{evaluation_mode}.csv"
    inference_xlsx_path = output_dir / f"inference_metrics_per_scenario__{evaluation_mode}.xlsx"

    results_df.to_csv(inference_csv_path, index=False)
    print("CSV per skenario disimpan ke :", inference_csv_path)

    try:
        results_df.to_excel(inference_xlsx_path, index=False)
        print("Excel per skenario disimpan ke:", inference_xlsx_path)
    except Exception as err:
        print("Excel per skenario tidak berhasil disimpan. CSV tetap tersedia.")
        print("Error:", err)

    metric_columns = ["accuracy", "precision", "recall", "f1_score", "pd", "pfa", "youden_index"]

    summary_by_model = results_df.groupby("model_type")[metric_columns].mean().reset_index()
    summary_by_pu = results_df.groupby(["model_type", "receiver_band"])[metric_columns].mean().reset_index()
    summary_by_type = results_df.groupby(["model_type", "scenario_type"])[metric_columns].mean().reset_index()
    summary_by_pu_type = results_df.groupby(["model_type", "receiver_band", "scenario_type"])[metric_columns].mean().reset_index()

    summary_by_model.to_csv(output_dir / f"inference_summary_by_model__{evaluation_mode}.csv", index=False)
    summary_by_pu.to_csv(output_dir / f"inference_summary_by_pu__{evaluation_mode}.csv", index=False)
    summary_by_type.to_csv(output_dir / f"inference_summary_by_scenario_type__{evaluation_mode}.csv", index=False)
    summary_by_pu_type.to_csv(output_dir / f"inference_summary_by_pu_and_scenario_type__{evaluation_mode}.csv", index=False)

    summary_xlsx_path = output_dir / f"inference_metrics_and_summaries__{evaluation_mode}.xlsx"
    try:
        with pd.ExcelWriter(summary_xlsx_path) as writer:
            results_df.to_excel(writer, sheet_name="per_scenario", index=False)
            summary_by_model.to_excel(writer, sheet_name="summary_by_model", index=False)
            summary_by_pu.to_excel(writer, sheet_name="summary_by_pu", index=False)
            summary_by_type.to_excel(writer, sheet_name="summary_by_type", index=False)
            summary_by_pu_type.to_excel(writer, sheet_name="summary_by_pu_type", index=False)
        print("Excel gabungan disimpan ke:", summary_xlsx_path)
    except Exception as err:
        print("Excel gabungan tidak berhasil disimpan. Seluruh CSV summary tetap tersedia.")
        print("Error:", err)

    print("\nRata-rata per model:")
    display(summary_by_model)

    print("\nRata-rata per PU:")
    display(summary_by_pu)

    print("\nRata-rata per jenis skenario:")
    display(summary_by_type)

    print("\nRata-rata per PU dan jenis skenario:")
    display(summary_by_pu_type)

    return {
        "per_scenario": results_df,
        "summary_by_model": summary_by_model,
        "summary_by_pu": summary_by_pu,
        "summary_by_type": summary_by_type,
        "summary_by_pu_type": summary_by_pu_type,
    }


## 10. Grid Search pada Dataset Gabungan Invariant CNN-only


In [18]:
CNN_PARAM_GRID = {
    "filters": [(64, 128)],
    "kernel_size": [3],
    "dense_units": [256],
    "pooling": ["gap"],
    "optimizer_name": ["adam"],
    "learning_rate": [1e-3],
    "epochs": [35],
    "batch_size": [256],
}


In [19]:
# Cek estimasi jumlah training grid search.
print_grid_search_plan(minutes_per_training=2.0)

RUN_GRID_SEARCH_CNN = True
RUN_TEST_INFERENCE_PER_SCENARIO = True
THRESHOLD = 0.5

all_inference_results = []

if RUN_GRID_SEARCH_CNN:
    manifest = load_manifest()

    cnn_files = select_dataset_files(
        manifest,
        model_type="cnn",
        scenario_types=[SCENARIO_TYPE_INVARIANT],
    )

    X_cnn_all, y_cnn_all, cnn_metadata = load_and_concat(cnn_files)
    print("CNN all dataset:", X_cnn_all.shape, y_cnn_all.shape)

    cnn_results_df, best_cnn = manual_grid_search(
        X_cnn_all,
        y_cnn_all,
        model_type="cnn",
        param_grid=CNN_PARAM_GRID,
    )

    cnn_results_path = OUTPUT_DIR / "grid_search_results_cnn__invariant_noise_statistics_and_fading.csv"
    cnn_results_df.to_csv(cnn_results_path, index=False)
    print("Hasil grid search CNN disimpan ke:", cnn_results_path)

    cnn_eval = evaluate_best_model(best_cnn, X_cnn_all, y_cnn_all)

    cnn_model_path = OUTPUT_DIR / "best_cnn__invariant_noise_statistics_and_fading.keras"
    best_cnn["model"].save(cnn_model_path)
    print("Model terbaik CNN disimpan ke:", cnn_model_path)

    cnn_eval_to_save = {k: v for k, v in cnn_eval.items() if k not in {"y_pred", "y_prob"}}
    with open(OUTPUT_DIR / "test_evaluation_cnn__invariant_noise_statistics_and_fading.json", "w", encoding="utf-8") as f:
        json.dump(cnn_eval_to_save, f, indent=2)

    if RUN_TEST_INFERENCE_PER_SCENARIO:
        print("\nMembuat metrik test per skenario untuk CNN dengan idx_test training yang sama...")
        cnn_table = build_dataset_table(manifest, model_type="cnn")
        cnn_ranges_df = collect_dataset_ranges_from_table(cnn_table)
        cnn_inference_df = evaluate_model_per_scenario_from_training_split(
            model=best_cnn["model"],
            dataset_table=cnn_table,
            ranges_df=cnn_ranges_df,
            split_indices=best_cnn["split_indices"],
            threshold=THRESHOLD,
        )
        all_inference_results.append(cnn_inference_df)
        display(cnn_inference_df.head())

    del X_cnn_all, y_cnn_all, cnn_metadata, best_cnn
    gc.collect()
    tf.keras.backend.clear_session()

if RUN_TEST_INFERENCE_PER_SCENARIO and all_inference_results:
    inference_results_df = pd.concat(all_inference_results, ignore_index=True)
    inference_exports = export_inference_results(
        inference_results_df,
        evaluation_mode="training_test_split_invariant_cnn_only",
        output_dir=OUTPUT_DIR,
    )
    display(inference_results_df)
elif RUN_TEST_INFERENCE_PER_SCENARIO:
    print("Tidak ada hasil inference karena grid search tidak dijalankan.")


Rencana grid search dataset invariant CNN-only:
  CNN      : 1 training
  Total    : 1 training
  Estimasi : 2.0 menit / 0.03 jam
CNN all dataset: (24000, 12, 12, 1) (24000,)
Pembagian data:
  Train      : 16,799 data (70.0%)
  Validation : 2,401 data (10.0%)
  Test       : 4,800 data (20.0%)
Split indices cnn disimpan ke: dataset\generated_datasets\global_split_indices_cnn.npz

=== Grid run 1/1 | cnn ===
{'filters': (64, 128), 'kernel_size': 3, 'dense_units': 256, 'pooling': 'gap', 'optimizer_name': 'adam', 'learning_rate': 0.001, 'epochs': 35, 'batch_size': 256}
Epoch 1/35
66/66 [==============================] - 2s 27ms/step - loss: 0.3425 - accuracy: 0.8618 - val_loss: 0.0618 - val_accuracy: 0.9938
Epoch 2/35
66/66 [==============================] - 1s 20ms/step - loss: 0.0267 - accuracy: 0.9974 - val_loss: 0.0093 - val_accuracy: 1.0000
Epoch 3/35
66/66 [==============================] - 1s 21ms/step - loss: 0.0082 - accuracy: 0.9992 - val_loss: 0.0053 - val_accuracy: 0.9988
Epoch 

,model_type,receiver_band,scenario_type,scenario_name,evaluation_mode,threshold,n_total_file,n_eval,dataset_path,accuracy,...,f1_score,pd,pfa,youden_index,tn,fp,fn,tp,support_0_noise_only,support_1_pu_noise
0,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_2db,training_test_split,0.5,2000,409,dataset\generated_datasets\pu1_low__invariant_...,1.0,...,1.0,1.0,0.0,1.0,199,0,0,210,199,210
1,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_4db,training_test_split,0.5,2000,396,dataset\generated_datasets\pu1_low__invariant_...,1.0,...,1.0,1.0,0.0,1.0,195,0,0,201,195,201
2,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_6db,training_test_split,0.5,2000,380,dataset\generated_datasets\pu1_low__invariant_...,1.0,...,1.0,1.0,0.0,1.0,186,0,0,194,186,194
3,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_8db,training_test_split,0.5,2000,391,dataset\generated_datasets\pu1_low__invariant_...,1.0,...,1.0,1.0,0.0,1.0,195,0,0,196,195,196
4,cnn,pu2_mid,invariant_noise_statistics_and_fading,path_loss_minus_2db,training_test_split,0.5,2000,399,dataset\generated_datasets\pu2_mid__invariant_...,1.0,...,1.0,1.0,0.0,1.0,196,0,0,203,196,203


CSV per skenario disimpan ke : dataset\generated_datasets\inference_metrics_per_scenario__training_test_split_invariant_cnn_only.csv
Excel per skenario disimpan ke: dataset\generated_datasets\inference_metrics_per_scenario__training_test_split_invariant_cnn_only.xlsx
Excel gabungan disimpan ke: dataset\generated_datasets\inference_metrics_and_summaries__training_test_split_invariant_cnn_only.xlsx

Rata-rata per model:


,model_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,0.999792,1.0,0.999545,0.999772,0.999545,0.0,0.999545



Rata-rata per PU:


...


Rata-rata per jenis skenario:


,model_type,scenario_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,invariant_noise_statistics_and_fading,0.999792,1.0,0.999545,0.999772,0.999545,0.0,0.999545



Rata-rata per PU dan jenis skenario:


,model_type,receiver_band,scenario_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,pu1_low,invariant_noise_statistics_and_fading,1.000000,1.0,1.000000,1.000000,1.000000,0.0,1.000000
1,cnn,pu2_mid,invariant_noise_statistics_and_fading,1.000000,1.0,1.000000,1.000000,1.000000,0.0,1.000000
2,cnn,pu3_high,invariant_noise_statistics_and_fading,0.999377,1.0,0.998634,0.999315,0.998634,0.0,0.998634


,model_type,receiver_band,scenario_type,scenario_name,evaluation_mode,threshold,n_total_file,n_eval,dataset_path,accuracy,...,f1_score,pd,pfa,youden_index,tn,fp,fn,tp,support_0_noise_only,support_1_pu_noise
0,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_2db,training_test_split,0.5,2000,409,dataset\generated_datasets\pu1_low__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,199,0,0,210,199,210
1,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_4db,training_test_split,0.5,2000,396,dataset\generated_datasets\pu1_low__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,195,0,0,201,195,201
2,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_6db,training_test_split,0.5,2000,380,dataset\generated_datasets\pu1_low__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,186,0,0,194,186,194
3,cnn,pu1_low,invariant_noise_statistics_and_fading,path_loss_minus_8db,training_test_split,0.5,2000,391,dataset\generated_datasets\pu1_low__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,195,0,0,196,195,196
4,cnn,pu2_mid,invariant_noise_statistics_and_fading,path_loss_minus_2db,training_test_split,0.5,2000,399,dataset\generated_datasets\pu2_mid__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,196,0,0,203,196,203
5,cnn,pu2_mid,invariant_noise_statistics_and_fading,path_loss_minus_4db,training_test_split,0.5,2000,406,dataset\generated_datasets\pu2_mid__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,197,0,0,209,197,209
6,cnn,pu2_mid,invariant_noise_statistics_and_fading,path_loss_minus_6db,training_test_split,0.5,2000,399,dataset\generated_datasets\pu2_mid__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,203,0,0,196,203,196
7,cnn,pu2_mid,invariant_noise_statistics_and_fading,path_loss_minus_8db,training_test_split,0.5,2000,436,dataset\generated_datasets\pu2_mid__invariant_...,1.000000,...,1.00000,1.000000,0.0,1.000000,217,0,0,219,217,219
8,cnn,pu3_high,invariant_noise_statistics_and_fading,path_loss_minus_2db,training_test_split,0.5,2000,381,dataset\generated_datasets\pu3_high__invariant...,1.000000,...,1.00000,1.000000,0.0,1.000000,190,0,0,191,190,191
9,cnn,pu3_high,invariant_noise_statistics_and_fading,path_loss_minus_4db,training_test_split,0.5,2000,396,dataset\generated_datasets\pu3_high__invariant...,1.000000,...,1.00000,1.000000,0.0,1.000000,198,0,0,198,198,198


## 11. Contoh Grid Search Per Kelompok Skenario

Bagian ini opsional. Untuk notebook ini, skenario utama sudah dibatasi pada `invariant_noise_statistics_and_fading`.


In [17]:
# Contoh filter jika ingin melatih hanya PU tertentu.
# manifest = load_manifest()
# files_pu1 = select_dataset_files(
#     manifest,
#     model_type="cnn",
#     receiver_bands=["pu1_low"],
#     scenario_types=[SCENARIO_TYPE_INVARIANT],
# )
# X_pu1, y_pu1, _ = load_and_concat(files_pu1)
# results_df, best = manual_grid_search(X_pu1, y_pu1, "cnn", CNN_PARAM_GRID)


## Ringkasan Cara Pakai

Notebook ini sudah dibuat untuk **sekali run** dari atas ke bawah.

Sebelum menjalankan:

1. Pastikan `BASE_DIR` sudah sesuai.
2. Jalankan semua cell dengan `Runtime > Run all`.
3. Notebook ini **tidak membutuhkan file fading CSV**, karena fading/path loss dibuat langsung dari nilai -2, -4, -6, dan -8 dB.

Default output:

```text
generated_datasets/
├── 12 file .npz CNN
├── manifest_invariant_cnn_only.json
├── global_split_indices_cnn.npz
├── grid_search_results_cnn__invariant_noise_statistics_and_fading.csv
├── test_evaluation_cnn__invariant_noise_statistics_and_fading.json
├── best_cnn__invariant_noise_statistics_and_fading.keras
├── inference_metrics_per_scenario__training_test_split_invariant_cnn_only.csv
├── inference_summary_by_model__training_test_split_invariant_cnn_only.csv
├── inference_summary_by_pu__training_test_split_invariant_cnn_only.csv
├── inference_summary_by_scenario_type__training_test_split_invariant_cnn_only.csv
└── inference_summary_by_pu_and_scenario_type__training_test_split_invariant_cnn_only.csv
```

Total data yang dihasilkan:

```text
3 PU × 4 path loss × 2000 data = 24.000 data CNN
```
